# 03 Learners

**Stage 2.** Tune one set of LightGBM settings per outcome, then fit the response baseline and the S, T and X learners and write the scores file.

**One set of settings for every model.** We tune once, by 3 fold cross validation on training rows only, using a small grid of 6 settings and log loss. The winner is shared by the baseline and all three learners. Any difference between models then comes from the strategy, not from how much tuning each one got. The tuning is done at 1M rows and reused at other sizes.

**The scores file.** One row per held out user, with the real treatment and outcomes, and one column per model and outcome: `{model}_{outcome}`. Notebook 04 only reads these files and never fits anything.

In [1]:
import json, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent
RESULTS, REPORT = ROOT / "results", ROOT / "report"
RESULTS.mkdir(exist_ok=True); REPORT.mkdir(exist_ok=True)

from uplift.data import FEATURES, TREATMENT, OUTCOMES, make_splits, take, treated_share_ok, TRAIN_SIZES
from uplift import metrics as m

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
NAMES = {"response": "Response baseline", "s": "S-learner", "t": "T-learner", "x": "X-learner"}
COLORS = {"response": "#7a7a7a", "s": "#1c7293", "t": "#f2a93b", "x": "#0b3d5c"}
from uplift.pipeline import tune_base, build_scores
kept = json.load(open(RESULTS / "kept_features.json"))
df = pd.read_parquet(ROOT / "data" / "criteo.parquet")
test_ids, pool_ids = make_splits(len(df))
dev = take(df, pool_ids[:1_000_000])
print("features:", kept)

features: ['f0', 'f2', 'f3', 'f4', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11']


In [2]:
best, tables = {}, {}
for outcome in OUTCOMES:
    best[outcome], tables[outcome] = tune_base(dev, kept, outcome)
    tables[outcome].round(6).to_csv(RESULTS / f"tuning_{outcome}.csv", index=False)
    print(outcome, "best:", best[outcome])
json.dump(best, open(RESULTS / "best_params.json", "w"), indent=1)
tables["visit"].round(6)

visit best: {'num_leaves': 15, 'learning_rate': 0.05, 'n_estimators': 200, 'min_child_samples': 200}


conversion best: {'num_leaves': 15, 'learning_rate': 0.05, 'n_estimators': 200, 'min_child_samples': 200}


,num_leaves,learning_rate,n_estimators,min_child_samples,cv_logloss
0,15,0.05,200,200,0.104805
1,15,0.05,400,200,0.104964
2,31,0.05,300,100,0.105318
3,31,0.03,500,300,0.105152
4,63,0.03,400,300,0.105696
5,63,0.02,600,500,0.105497


## Fit and score at each training size

For each nested training size we fit 8 models per outcome: the response baseline, the S-learner (1 model), the T-learner (2 models) and the X-learner (4 models). All are scored on the same 2M held out rows. The 1M file is the development run, the 5M file gives the headline numbers, and 100k and 500k feed the data size curve.

In [3]:
import time
LABELS = {100_000: "100k", 500_000: "500k", 1_000_000: "1m", 5_000_000: "5m"}
for n in TRAIN_SIZES:
    t0 = time.time()
    scores = build_scores(df, pool_ids[:n], test_ids, kept, best)
    scores.to_parquet(RESULTS / f"scores_{LABELS[n]}.parquet")
    print(f"train {n:>9,} rows -> scores_{LABELS[n]}.parquet {scores.shape} in {time.time() - t0:.0f}s")
scores.head()

train   100,000 rows -> scores_100k.parquet (2000000, 11) in 60s


train   500,000 rows -> scores_500k.parquet (2000000, 11) in 58s


train 1,000,000 rows -> scores_1m.parquet (2000000, 11) in 103s


train 5,000,000 rows -> scores_5m.parquet (2000000, 11) in 165s


,treatment,visit,conversion,response_visit,s_visit,t_visit,x_visit,response_conversion,s_conversion,t_conversion,x_conversion
row_id,,,,,,,,,,,
4290029,1,0,0,0.000736,0.000057,0.000204,0.000383,0.000073,0.000024,0.000056,0.000105
3826474,1,1,0,0.285788,0.057680,0.080194,0.047501,0.005215,0.002468,0.001339,0.001651
484259,1,0,0,0.000986,0.000109,0.000239,0.000410,0.000098,0.000031,0.000088,0.000105
13866927,1,0,0,0.209659,0.030492,0.025709,0.032208,0.011103,0.005865,0.003453,0.006751
3811898,1,0,0,0.000962,0.000153,0.000248,0.000476,0.000096,0.000027,0.000058,0.000105


In [4]:
# sanity checks on the largest scores file
s = pd.read_parquet(RESULTS / "scores_5m.parquet")
assert s.isna().sum().sum() == 0 and len(s) == 2_000_000
print(s.columns.tolist())
s.filter(like="_visit").describe().T.round(5)

['treatment', 'visit', 'conversion', 'response_visit', 's_visit', 't_visit', 'x_visit', 'response_conversion', 's_conversion', 't_conversion', 'x_conversion']


,count,mean,std,min,25%,50%,75%,max
response_visit,2000000.0,0.04695,0.12152,0.00059,0.00085,0.00365,0.01815,0.87300
s_visit,2000000.0,0.00704,0.02043,-0.00928,0.00009,0.00043,0.00286,0.22230
t_visit,2000000.0,0.00769,0.02503,-0.12098,0.00013,0.00042,0.00277,0.37574
x_visit,2000000.0,0.00768,0.02398,-0.10695,0.00040,0.00101,0.00279,0.35576
